# Flash Loans — Where The Money Goes Missing

Books One and Two built machinery. This one spends it.

Notebook 6 built two contracts on the chain we already had: a swap pool and a lending protocol. The lending rule asked the puddle what ETH was worth. That is the bug. This notebook is the shove.

Featuring a robbery in which nobody breaks a single rule.

**KEY INSIGHT.** How can a protocol execute every instruction correctly, revert properly when a step fails, run on a chain with flawless consensus and unbroken cryptography — and still lose all its money when every single step succeeds?

**Scope:** a deterministic teaching model, not a recipe for attacking a real protocol. No fees, no liquidity providers, no lawyers. The hashes will stay valid throughout. The money vanishes anyway.

The toy model is ours. The mechanisms are not — see **Sources** at the end.


## Recap

If you remember only one line from notebook 6 going into this one, make it: a smart contract's **code** cannot be changed, but the **numbers** it operates on move constantly. Every heist here is somebody moving the numbers.

We **import** `AMMPool`, `LendingProtocol`, and `submit_call` from [`blockchain_lib/contracts.py`](../blockchain_lib/contracts.py). We do not reinvent them. `FlashLoanProvider` and `MedianOracle` arrive when Act Two needs them. What notebook 6 did *not* do is rob anyone. The 40 ETH boulder stayed a thought experiment.

The cast, none of them malicious, none of them buggy:

- `AMMPool` — a thin pool holding 50 ETH and $100,000. It reports the ratio of its own two piles. That is its entire job and it does it honestly.
- `LendingProtocol` — liquidates any loan below 1.5x. Reads the price from the AMM, because the AMM is right there.
- `FlashLoanProvider` — will lend you any amount of ETH with zero collateral, on one condition: give it back before this transaction ends.

And the victim: $12,000 against 10 ETH. Ratio 1.67. Not doing anything risky. About to be robbed anyway.

Three acts. First Alice uses a suitcase of cash she already owns. Then she rents the suitcase. Then the heist fails, and the failure is the interesting part. **Included is not the same as succeeded.**


In [ ]:
import random

from blockchain_lib.contracts import AMMPool, LendingProtocol, submit_call
from blockchain_lib.mempool import Network, Transaction
from blockchain_lib.pos import Blockchain, Validator

# submit_call: shout it, stamp it, fill in the form. Notebook 6's notary helper.
# Two of these still make two transactions. That is Act One: you had to be rich first.


## Act One: Alice does it the old-fashioned way

Before flash loans, this attack still worked — it just required being rich first. Alice owns **40 ETH**. She uses it in two ordinary transactions. No lying courier required. The pool will tell the truth about *itself*, and that truth will be a terrible proxy for "the market."

Between the dump and the liquidation the cheap price sits on-chain, in public, the way notebook 5 said a waiting room works. In this toy nobody snipes it. On a real network, somebody might.

This is not a flash loan. Alice had to be rich *before* lunch.

> Pause and predict: after paying the $12,000 debt and buying ETH back, how much of Alice's original 40 ETH remains — and did she need to own those 40 ETH the whole time?


In [1]:
# Fresh puddle, same 50 ETH / $100,000 as notebook 6. We are about to soak everyone nearby.
nodes = ["Node A", "Alice-Node", "Bob-Node", "Farid-Node"]
validators = [
    Validator("Node A", 100),
    Validator("Alice-Node", 80),
    Validator("Bob-Node", 70),
    Validator("Farid-Node", 50),
]
owned_network = Network(nodes, random.Random(7))
owned_chain = Blockchain(validators)

# Same two contracts, neither buggy. The lending rule still asks the puddle about the ocean.
owned_pool = AMMPool("amm-thin.eth", 50.0, 100_000.0)
owned_lending = LendingProtocol("lending-thin.eth", owned_pool)
owned_lending.call(
    "open_loan", borrower="victim", collateral_eth=10.0, debt_usd=12_000.0
)

# TX1: Alice dumps 40 ETH she already owns. The cheap price is now public. Crime scene, open floor plan.
alice_eth = 40.0
usd_from_dump, dump_tx, dump_block = submit_call(
    owned_network,
    owned_chain,
    "Alice-Node",
    "tx-dump-40eth",
    owned_pool,
    "swap_eth_for_usd",
    eth_in=alice_eth,
)
manipulated_price = owned_pool.spot_price
victim_ratio = owned_lending.collateral_ratio(owned_lending.loans["victim"])

# TX2: liquidate at that public photograph, then buy ETH back at the price she herself created.
seized, liq_tx, liq_block = submit_call(
    owned_network,
    owned_chain,
    "Alice-Node",
    "tx-liquidate-victim",
    owned_lending,
    "liquidate",
    borrower="victim",
)
usd_for_buyback = usd_from_dump - seized.debt_usd
eth_bought_back = owned_pool.call("swap_usd_for_eth", usd_in=usd_for_buyback)
alice_final_eth = eth_bought_back + seized.collateral_eth
profit_eth = alice_final_eth - alice_eth

print(f"{dump_block.proposer} included {dump_tx.tx_id} in Block #{dump_block.index}")
print(
    f"DUMP: Alice sells {alice_eth:.2f} ETH of her own for ${usd_from_dump:,.2f}; "
    f"AMM price becomes ${manipulated_price:,.2f}/ETH."
)
print(
    f"The cheap puddle is now public. Victim ratio at this price: {victim_ratio:.2f}."
)
print(f"{liq_block.proposer} included {liq_tx.tx_id} in Block #{liq_block.index}")
print(
    f"LIQUIDATE: pay ${seized.debt_usd:,.2f} debt and seize "
    f"{seized.collateral_eth:.2f} ETH."
)
print(
    f"BUY BACK: remaining ${usd_for_buyback:,.2f} buys {eth_bought_back:.2f} ETH."
)
print(
    f"Alice started with {alice_eth:.2f} ETH, ends with {alice_final_eth:.2f} ETH, "
    f"profit {profit_eth:.2f} ETH."
)
print("She had to own the 40 ETH already. The bug is the price source, not the financing.")


Alice-Node included tx-dump-40eth in Block #1
DUMP: Alice sells 40.00 ETH of her own for $44,444.44; AMM price becomes $617.28/ETH.
The cheap puddle is now public. Victim ratio at this price: 0.51.
Alice-Node included tx-liquidate-victim in Block #2
LIQUIDATE: pay $12,000.00 debt and seize 10.00 ETH.
BUY BACK: remaining $32,444.44 buys 33.18 ETH.
Alice started with 40.00 ETH, ends with 43.18 ETH, profit 3.18 ETH.
She had to own the 40 ETH already. The bug is the price source, not the financing.


**Read the result.** Alice sells 40 ETH into a pool that only had 50. Price collapses 69% — not because ETH crashed, but because she displaced most of the puddle. At $617/ETH the victim's 10 ETH is "worth" $6,172 against $12,000. Ratio 0.51. The protocol, following its rules correctly, marks them liquidatable.

She pays the $12,000 debt, seizes 10 ETH, and uses the remaining $32,444 to buy ETH back — at the depressed price she herself created. Net: **3.18 ETH** of pure profit, extracted from someone who never did anything wrong.

Read the last beat: by the time anyone looked at a price chart, the spike had mostly healed. The trail tidies itself on the way out. That is why oracle manipulation is so hard to spot after the fact.

Two things before we rent the suitcase:

1. **The bug is the oracle choice.** Alice did not break arithmetic. She fed the lending contract a local ratio and it believed her.
2. **The financing is still a suitcase of cash.** That is roughly $80,000 sitting idle, which meaningfully limits how many people can attempt this.


## Act Two: the same heist, but Alice is broke

Act One had a barrier to entry. A flash loan removes that barrier completely.

The idea sounds insane the first time you hear it: a contract will lend you unlimited money with zero collateral, no credit check, and no questions asked. The catch is a single condition — the money must be back by the end of this transaction. If it is not, the entire transaction reverts and it is as though nothing happened.

**ANALOGY.** A bank that will hand you ten million dollars, no questions asked, provided you return it before you finish walking through the lobby. You cannot steal it, because if you walk out the door still holding it, the universe rewinds to before you were handed it. So the loan is not risky at all — which is exactly why it can be uncollateralised, and exactly why anyone can have one.

Five operations. One transaction. Either all of them happen, or none of them do.

| Stage | What changes | Why it matters |
| --- | --- | --- |
| Borrow | attacker temporarily receives ETH | capital is available only for this walk through the lobby |
| Dump | AMM spot price falls | the lending rule sees a misleading input |
| Liquidate | attacker pays debt and receives collateral | the vulnerable rule follows its price source |
| Buy back and repay | borrowed principal returns to the lender | any remaining ETH belongs to the attacker |

> Pause and predict: after paying the $12,000 debt and reversing the dump, how much ETH remains once the 40 ETH principal is repaid? (You have seen this movie. The ending should look familiar.)


## Flash loans and the heist

This toy provider charges no fee. It snapshots everything it touches and restores that snapshot if anything fails. Only ETH left after repayment counts as attacker profit.

The next cell adds the flash-loan types from [`blockchain_lib/flash_loan.py`](../blockchain_lib/flash_loan.py), plus the oracle helpers Act Two's defence will need. `AMMPool` and `LendingProtocol` are already imported. Open the library file if you want the snapshot/rollback loop. Here is the new cast:

- `FlashLoanProvider.execute` — lend, run an action, repay or rewind.
- `run_flash_attack` — borrow, dump, liquidate, buy back (the walk through the lobby).
- `TransactionReceipt` — inclusion is not the same as success. Write that down again.

The provider has 1,000 ETH; the attacker borrows 40. Same puddle as notebook 6. Same victim.

> Pause and predict: if SketchyGuy-Node originates the attack, can a peer who has not received it include it?


In [5]:
from blockchain_lib.contracts import (
    MedianOracle,
    PositionNotLiquidatableError,
    PriceReport,
)
from blockchain_lib.flash_loan import (
    AttackTrace,
    FlashLoanProvider,
    TransactionReceipt,
    run_flash_attack,
)

# New for Act Two. AMMPool, LendingProtocol, Network, and the chain classes stay from the first import cell.


### Submit the heist as a `Transaction`

Two cells, on purpose. This one only does notebook 5: gossip `tx-flash-attack`, refuse inclusion from a node that never heard it, then `include` from SketchyGuy-Node. No pool moves yet. The next cell runs `FlashLoanProvider.execute` against that already-stamped receipt.

On a real chain, inclusion and execution share a block. Here they are two cells so you can see a reverted call still leave a receipt. The block does not care about your feelings.

Walk this cell as: new validators, shout the payload, a peer with an empty inbox cannot stamp it, SketchyGuy-Node can — because the tx sits in *their* mempool. There is no THE mempool. There never was.


In [7]:
# Fresh chain. Same crime, rented boulder. Alice does not own any of it.
nodes = ["Node A", "SketchyGuy-Node", "Emma-Node", "Farid-Node"]
validators = [
    Validator("Node A", 100),
    Validator("SketchyGuy-Node", 80),
    Validator("Emma-Node", 70),
    Validator("Farid-Node", 50),
]
network = Network(nodes, random.Random(7))
chain = Blockchain(validators)
receipts: list[TransactionReceipt] = []

# Stamp first, execute later. A receipt can outlive a revert. Inclusion is not success.
attack_tx = Transaction("tx-flash-attack", "Flash-loan oracle manipulation")
network.broadcast(attack_tx, origin="SketchyGuy-Node")

print("Each node has its own mempool after the attack is gossiped:")
for node in nodes:
    print(f"  {node}: {network.mempool_ids(node)}")

# Notebook 5 still applies: you cannot stamp a rumour you never heard. SketchyGuy was in the group chat.
missing = [name for name in nodes if network.get(name, attack_tx.tx_id) is None]
if missing:
    try:
        network.include(missing[0], attack_tx.tx_id, chain)
    except ValueError as error:
        print(f"\n{error}")

included_attack, attack_block = network.include(
    "SketchyGuy-Node", attack_tx.tx_id, chain
)
print(
    f"\n{attack_block.proposer} included {included_attack.tx_id} "
    f"in Block #{attack_block.index}."
)
print(
    "Mempools after inclusion: "
    + ", ".join(f"{name}={network.mempool_ids(name)}" for name in nodes)
)


Each node has its own mempool after the attack is gossiped:
  Node A: ['tx-flash-attack']
  SketchyGuy-Node: ['tx-flash-attack']
  Emma-Node: ['tx-flash-attack']
  Farid-Node: []

Farid-Node cannot include tx-flash-attack: it is not in their local mempool.

SketchyGuy-Node included tx-flash-attack in Block #1.
Mempools after inclusion: Node A=[], SketchyGuy-Node=[], Emma-Node=[], Farid-Node=[]


### Run `FlashLoanProvider.execute`

Same puddle, same victim, same 40 ETH — but the ETH is rented. `execute` snapshots the pool, the loans, and the provider, then calls `run_flash_attack`. If repayment succeeds the snapshot is discarded. Profit is whatever ETH remains after the 40 ETH principal returns.

That is the same arithmetic as Act One. The financing changed. The bug did not.

**KEY INSIGHT.** Flash loans did not create this vulnerability. The bug was the oracle choice, and it was there in Act One with Alice's own money. What flash loans changed is **who** can exploit it: previously, only people with $80,000 lying around. Now, anyone with a laptop and a gas fee. They democratised an attack that already existed, which is a genuinely novel thing for a financial primitive to do.


In [8]:
# Same puddle, same victim. The boulder is on loan until we finish walking through the lobby.
attack_pool = AMMPool("amm.eth", 50.0, 100_000.0)
attack_protocol = LendingProtocol("lending.eth", attack_pool)
attack_protocol.open_loan("victim", 10.0, 12_000.0)
provider = FlashLoanProvider(1_000.0)

# Five steps, one transaction. Either all of them happen, or the universe rewinds.
profit_eth, attack_trace = provider.execute(
    40.0,
    lambda borrowed_eth: run_flash_attack(
        borrowed_eth, attack_pool, attack_protocol, "victim"
    ),
    attack_pool,
    attack_protocol,
)
assert attack_trace.eth_before_repayment == (
    attack_trace.principal_repaid + profit_eth
)
assert provider.eth_available == 1_000.0

print("Attack transaction: COMMITTED")
print(f"Provider liquidity after repayment: {provider.eth_available:,.2f} ETH")
print(f"Attacker profit: {profit_eth:.2f} ETH")
print("Same 3.18 ETH as the owned-capital dump. Alice did not need to own the 40 ETH first.")

receipts.append(
    TransactionReceipt(
        included_attack,
        attack_block,
        "SUCCESS",
        310_000,
        "Attack state committed",
    )
)


STEP 1 — BORROW: 40.00 ETH arrives temporarily.
STEP 2 — DUMP: sell 40.00 ETH for $44,444.44; AMM price becomes $617.28/ETH.
STEP 3 — LIQUIDATE: victim ratio is 0.51; pay $12,000.00 debt and seize 10.00 ETH.
STEP 4 — BUY BACK: remaining $32,444.44 buys back 33.18 ETH; attacker holds 43.18 ETH.
STEP 5 — REPAY: return 40.00 ETH principal to the provider.
Attack transaction: COMMITTED
Provider liquidity after repayment: 1,000.00 ETH
Attacker profit: 3.18 ETH
Same 3.18 ETH as the owned-capital dump. Alice did not need to own the 40 ETH first.


**Read the result.** Identical profit. **3.18 ETH.** Zero starting capital. The provider ends with exactly what it started with and is perfectly happy.

Every step followed the rules. The rules asked the puddle for the price of the ocean. SketchyGuy-Node could `include` the payload only because `broadcast` had already put it in that node's mempool.

`revert()` did not fire. It has no opinion about whether an outcome is fair, intended, or morally acceptable. Every step passed every check — because the checks were performed against a price that had been legitimately, honestly, and catastrophically manipulated one instruction earlier. Nothing failed. So nothing reverted. So it committed, permanently, exactly like any honest transaction.


## Act Three: it fails, and the failure is instructive

Change exactly one thing: Victim2 has 40 ETH of collateral instead of 10, against the same $12,000 debt. Even after the dump, they should survive.

Same `Transaction` path as before: `broadcast`, then `include`, **then** `execute`. The block is appended before liquidation fails. That is the point. The receipt will say `REVERTED`; the chain will still be longer by one. Atomicity refuses to leave a half-finished *world* around. It does not unwrite the attempt.

> Pause and predict: after liquidation fails, which values should look exactly as they did before the flash loan?


In [11]:
# Inclusion still happens. The notary stamped the attempt. Feelings are not a field in the block header.
failed_tx = Transaction("tx-flash-victim2", "Flash-loan against Victim2")
network.broadcast(failed_tx, origin="SketchyGuy-Node")
included_failed, failed_block = network.include(
    "SketchyGuy-Node", failed_tx.tx_id, chain
)
print(
    f"{failed_block.proposer} included {included_failed.tx_id} "
    f"in Block #{failed_block.index} (execution comes next)."
)

# Victim2 brought 40 ETH of collateral. The boulder is not big enough. The rule check will fire.
failed_pool = AMMPool("amm-victim2.eth", 50.0, 100_000.0)
failed_protocol = LendingProtocol("lending-victim2.eth", failed_pool)
failed_protocol.open_loan("Victim2", 40.0, 12_000.0)
failed_provider = FlashLoanProvider(1_000.0)
failed_before = (
    failed_pool.eth_reserve,
    failed_pool.usd_reserve,
    tuple(sorted(failed_protocol.loans)),
    failed_provider.eth_available,
)
failed_observation: dict[str, float] = {}


def failed_attack_action(amount_eth: float) -> tuple[float, AttackTrace]:
    """Dump, attempt liquidation, and refuse to pretend it succeeded.

    Args:
        amount_eth: Borrowed ETH sold into ``failed_pool``.

    Returns:
        Never returns on the happy path; liquidation should raise first.

    Raises:
        PositionNotLiquidatableError: If Victim2 is still healthy.
        RuntimeError: If liquidation unexpectedly succeeded.
    """
    failed_pool.swap_eth_for_usd(amount_eth)
    failed_observation["price"] = failed_pool.spot_price
    failed_observation["ratio"] = failed_protocol.collateral_ratio(
        failed_protocol.loans["Victim2"]
    )
    failed_protocol.liquidate("Victim2")
    raise RuntimeError("Liquidation should have raised first.")


try:
    failed_provider.execute(
        40.0, failed_attack_action, failed_pool, failed_protocol
    )
except PositionNotLiquidatableError:
    print(
        f"Attempted dump moved AMM price to "
        f"${failed_observation['price']:,.2f}/ETH."
    )
    print(
        f"Victim2 ratio after dump: {failed_observation['ratio']:.2f}; "
        "liquidation is rejected."
    )
    print("Attack transaction: REVERTED")

# Universe rewind: pool, loans, provider — all restored to the byte. As though the walk never happened.
failed_after = (
    failed_pool.eth_reserve,
    failed_pool.usd_reserve,
    tuple(sorted(failed_protocol.loans)),
    failed_provider.eth_available,
)
rollback_verified = failed_before == failed_after
print(f"Rollback complete: {rollback_verified}")

receipts.append(
    TransactionReceipt(
        included_failed,
        failed_block,
        "REVERTED",
        185_000,
        "No state change",
    )
)


SketchyGuy-Node included tx-flash-victim2 in Block #2 (execution comes next).
Attempted dump moved AMM price to $617.28/ETH.
Victim2 ratio after dump: 2.06; liquidation is rejected.
Attack transaction: REVERTED
Rollback complete: True


**Read the result.** The pool really did hit $617 during execution. The dump genuinely happened. Then liquidation raised an exception, and every trace of it was unwound — pool reserves, loan records, provider balance, all restored to the byte. Victim2's ratio was 2.06. Still above 1.50.

So why did Act Two not revert? Was it not also an attack?

**KEY INSIGHT.** `revert()` fires when a step **fails a rule check**. It has no opinion about fairness. In Act Two, every step passed every check. In Act Three, `liquidate()` raised. Both transactions were **included**. Notebook 5, again: inclusion is not success. The failed attempt is sitting on that chain forever, a permanent public receipt reading "someone tried this and it did not work."


## Defence: stop asking the puddle

Notebook 6 already gave us the fix: stop reading the price from a shovable puddle. Point the lending protocol at a `MedianOracle` instead and the attack dies — the pool still moves to $617, the protocol just does not care.

This dump is a **direct method call**, not a gossiped `Transaction`. We already practised `broadcast` / `include` on the heist. Here the only moving part we care about is the price source.

The dump still happens. The pool still looks seasick. The protocol simply stops taking medical advice from it.

A median of three sources survives one liar and fails against two. If those "independent" sources are themselves thin AMM pools, a flash loan large enough can move two of them in the same transaction. A write-up that says "we used multiple independent oracles" can be factually true and still miss the point: independence assumed the attacker could not shove several pools at once.

The reports are $2,005, $1,995, and $2,000.

> Pause and predict: after the dump pushes the AMM to about $617/ETH, will the protocol liquidate the original 10 ETH / $12,000 loan if its oracle is still near $2,000?


In [14]:
# Three reports, one median. No ML. Sort, pick the middle, ignore how loud the liar was.
defense_oracle = MedianOracle(
    [
        PriceReport("exchange-A", 2_005.0),
        PriceReport("exchange-B", 1_995.0),
        PriceReport("reference-feed", 2_000.0),
    ]
)
defense_pool = AMMPool("amm-defense.eth", 50.0, 100_000.0)
# price_source overrides the puddle. The dump can still soak everyone. The protocol just stops asking it.
defense_protocol = LendingProtocol(
    "lending-defense.eth", defense_pool, price_source=defense_oracle
)
defense_victim_loan = defense_protocol.open_loan("defense-victim", 10.0, 12_000.0)
defense_pool.swap_eth_for_usd(40.0)
defense_ratio = defense_protocol.collateral_ratio(defense_victim_loan)

try:
    defense_protocol.liquidate("defense-victim")
except PositionNotLiquidatableError:
    defense_rejected = True
else:
    defense_rejected = False


def defense_verdict(defense_rejected: bool) -> str:
    """Return a short label for whether the median-priced liquidation ran.

    Args:
        defense_rejected: True if ``liquidate`` raised.

    Returns:
        ``liquidation rejected`` or ``liquidation executed``.
    """
    return "liquidation rejected" if defense_rejected else "liquidation executed"


print(f"Manipulated AMM spot price: ${defense_pool.spot_price:,.2f}/ETH")
print(
    f"Median oracle price: ${defense_oracle.price:,.2f}/ETH; "
    f"protocol ratio: {defense_ratio:.2f}"
)
print(f"Defense result: {defense_verdict(defense_rejected)}")


Manipulated AMM spot price: $617.28/ETH
Median oracle price: $2,000.00/ETH; protocol ratio: 1.67
Defense result: liquidation rejected


**Read the result.** The AMM really moved to $617.28. The protocol read the median's $2,000 and rejected the healthy loan. Aggregation did not make the AMM honest — the AMM was always honest, and is still reporting $617 right now. The defence was changing which question you ask.

**REALITY BITE.** This is not hypothetical. Flash loan oracle manipulation has drained hundreds of millions of dollars from real protocols, repeatedly, for years, in attacks that were structurally identical to the one above. In almost every case the code was audited. The audits were looking for bugs. There were no bugs. There was a contract asking a puddle about the ocean, and an auditor who did not think to ask whether that was a sensible question.

Median feeds, TWAPs, deeper liquidity, freshness limits, circuit breakers: layers, each with a cost, each defeatable by an attacker willing to pay more. Anyone selling you an oracle solution with the word "solved" in the pitch has not read notebook 6.


## Included is not the same as succeeded

Notebook 5 separated origin, gossip receipt, inclusion, and confirmation. Add **execution status**. Both attack transactions were included — that is why we stamped them *before* `execute`. Only one left a state change. The other burned the teaching equivalent of gas and sat on the chain as a receipt of an attempt.

Atomicity prevents half-finished state. It does not unwrite the attempt. It does not audit the oracle. It does not have a category for "technically legal robbery."

> Pause and predict: which receipt was included but left no modeled state change?


In [17]:
print("Block | Transaction                      | Status   | Gas used | State effect")
for receipt in receipts:
    print(
        f"{receipt.block.index:>5} | {receipt.transaction.description:<32} "
        f"| {receipt.status:<8} | {receipt.gas_used:>8,} | {receipt.state_effect}"
    )
print("Included does not mean succeeded")
valid, message = chain.is_valid()
print(f"Chain valid? {valid} -- {message}")
print(f"Canonical length: {len(chain.chain)} (genesis + {len(receipts)} inclusions)")


Block | Transaction                      | Status   | Gas used | State effect
    1 | Flash-loan oracle manipulation   | SUCCESS  |  310,000 | Attack state committed
    2 | Flash-loan against Victim2       | REVERTED |  185,000 | No state change
Included does not mean succeeded
Chain valid? True -- Chain is valid.
Canonical length: 3 (genesis + 2 inclusions)


**Read the result.** Both payloads used notebook 5's `Network.broadcast` and `Network.include`. The heist committed. The Victim2 attempt reverted, but its receipt is still on the chain. Fork choice does not audit the oracle. The block does not care about your feelings.


## Takeaways

- **Mechanism:** you can shove a thin pool with capital you already own. **Not a guarantee:** you needed to be rich first, or that two separate transactions will stay unopposed. The crime scene cleans itself up.
- **Mechanism:** a flash loan makes huge capital available until the transaction ends. **Not a guarantee:** the lending rule was a good idea. Flash loans did not create the bug. They democratised it.
- **Mechanism:** a flash-loan attack is a `Transaction`: `broadcast`, then `include`. **Not a guarantee:** a missed gossip still lets you include it, or that inclusion means the heist succeeded.
- **Mechanism:** atomicity prevents half-finished state. **Not a guarantee:** a completed exploit of a bad price source gets rolled back out of fairness. `revert()` has no opinion about fairness.
- **Mechanism:** the same `MedianOracle` from notebook 6 stops this heist if the protocol actually reads it. **Not a guarantee:** aggregation is a silver bullet, or that your "independent" sources survive an attacker who can shove all of them at once.
- **Mechanism:** included is not the same as succeeded. **Not a guarantee:** a valid chain is a solvent protocol. Correct code, correct execution, correct consensus, and a completely wrong outcome.

Next: [8. stablecoins.ipynb](8.%20stablecoins.ipynb) — a token that claims to be a dollar, backed by a vault the chain cannot see. After notebook 6, that should make you deeply nervous. Correctly so.


## Sources

Atomic borrow-dump-liquidate-repay is a documented DeFi pattern, not a plot twist:

- Qin, K., Zhou, L., Livshits, B., & Gervais, A. (2021). [Attacking the DeFi Ecosystem with Flash Loans for Fun and Profit](https://arxiv.org/abs/2003.03810). Financial Cryptography. Same-transaction credit, oracle manipulation, and why atomicity does not make a bad price rule safe.
- Aave. [Flash Loans](https://aave.com/docs/developers/flash-loans). The production primitive: liquidity that must be returned before the transaction ends, or the whole call reverts.
- Adams, H., Zinsmeister, N., & Robinson, D. (2020). [Uniswap v2 Core](https://uniswap.org/whitepaper.pdf), flash swaps. Receive the asset first, pay it back in the same transaction — the same atomic suitcase.
